# FallVision 03: pose caching, features and 2-second windows

Pipeline step 1 of 3 after the data download. Everything is done by the scripts in `src/` through
`run_pipeline.py`; this notebook only finds the inputs, runs the stages and shows the results.

| Stage | Script | Output (in this notebook's Output, folder `work/`) |
|---|---|---|
| poses | `src/cache_poses.py` | YOLO11m-pose + ByteTrack skeletons of every split video (`work/poses`) |
| features | `src/preprocess.py` | 92 features per frame for the main person (`work/features`) |
| windows | `src/make_windows.py` | 2-s windows labelled normal / falling / fallen (`work/windows`) |

**Before running (Kaggle)**
- Inputs (*Add Input > Your Work > Notebooks*): the outputs of `01_download_omnifall_videos` and
  `02_download_gmdcsa24`.
- *Settings*: Accelerator **GPU T4 x1**, Internet **on**. *Add-ons > Secrets*: `GITHUB_TOKEN` on.
- Time: about 6-7 hours (pose caching of about 1,480 videos). Run it with **Save Version > Save & Run
  All (Commit)** so it runs in the background.

**Next:** `04_train_models` with this notebook's output as its input.

In [ ]:
# ---- settings: the only cell you may need to change ----
REPO_URL = "https://github.com/Prish1509/project_fallvision.git"   # your repository (public)
BRANCH = "main"
PUSH_RESULTS = True             # push the result files back to the repository (needs the GITHUB_TOKEN secret)
REPO = "/kaggle/working/project_fallvision"   # where the code is cloned in this notebook
WORK = "/kaggle/working/work"         # pose caches, features, windows, probabilities (kept in the Output)

## 1. Code and packages

In [ ]:
import os, shutil, subprocess, sys
os.makedirs("/kaggle/working", exist_ok=True)
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
import kaggle_helpers as kh

# the exact versions from requirements.txt of the packages Kaggle does not have (torch is Kaggle's own)
pins = [l.split("#")[0].strip() for l in open("requirements.txt")
        if l.split("==")[0].strip() in ("ultralytics", "ultralytics-thop", "lap")]
kh.run([sys.executable, "-m", "pip", "install", "-q", *pins])

import torch
print("code version:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - turn it on: Settings > Accelerator > GPU T4 x1")

## 2. Inputs: the videos and the pretrained pose model

In [ ]:
OMNI = kh.find_omnifall()
GMD = kh.find_gmd()
assert OMNI and GMD, "add the outputs of the two data notebooks as inputs (see the top of this notebook)"
kh.get_yolo("models/yolo11m-pose.pt")

## 3. Pose caching (YOLO11m-pose + ByteTrack on every video of `splits/videos.csv`)

In [ ]:
kh.run([sys.executable, "run_pipeline.py", "--omnifall-root", OMNI, "--gmd-root", GMD,
        "--work", WORK, "--from", "poses", "--to", "poses"])

In [ ]:
import csv, collections
index = list(csv.DictReader(open(f"{WORK}/poses/index.csv")))
ok = collections.Counter(r["split"] for r in index if r["status"] in ("cached", "already"))
bad = [r for r in index if r["status"] not in ("cached", "already")]
for split in ["train", "val", "test_gmdcsa24", "test_staged", "test_oops"]:
    print(f"{split:<15} videos with poses: {ok[split]}")
print(f"\nvideos with a problem: {len(bad)}")
for r in bad[:20]:
    print("  ", r["dataset"], r["path"], "->", r["status"])
if any(r["status"].startswith("skipped") for r in bad):
    print("WARNING: the time limit was reached before all videos were cached.")

## 4. Features and windows

In [ ]:
kh.run([sys.executable, "run_pipeline.py", "--work", WORK, "--from", "features", "--to", "windows"])

In [ ]:
print(open("results/windows_summary.md").read())
kh.show_change(REPO, "results/windows_summary.md")       # compared with the committed version

## 5. Save the summary files to GitHub

In [ ]:
if PUSH_RESULTS:
    kh.push(REPO, ["results/poses_index.csv", "results/features_manifest.csv", "results/windows_summary.md"],
            "Pipeline 03: pose caches, features and windows (summary files)", REPO_URL, BRANCH)
print("Done. Output kept by Kaggle:", WORK)